# NpuKit — MNIST tiny-ViT host smoke (PYNQ-Z2)

Geometry: resize **28→16**, patch **4** → 16 raw patches, pair-average → **T=8**, **D=8**.
(T=8 Softmax avoids a shipped-bit glue bug when `len==MAX_LEN`; RTL fix is in-tree for the next rebuild.)

| Where | What |
|-------|------|
| CPU | resize, patchify, pair-pool, pos add, mean-pool, 10-way head |
| FPGA | patch GEMM + 1-layer block (GEMM + glue) |

Seeded **random weights** — plumbing / ref-vs-board check, not trained accuracy.

Keep `npukit.bit`, `.hwh`, `npukit_vit_mnist.py`, `npukit_transformer.py`, `npukit_matmul.py` beside this notebook.

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_vit_mnist as vit

importlib.reload(vit)
print("IMG", vit.IMG, "PATCH", vit.PATCH, "T", vit.VIT_T, "D", vit.VIT_D)
print("BIT", BIT)

IMG 16 PATCH 4 T 8 D 8
BIT /home/xilinx/jupyter_notebooks/npukit.bit


## Offline ref path (no bitstream)

In [2]:
rc = vit.run_vit_smoke(bit_path=None, seed=0, n=2)
assert rc == 0
print("ref-only return", rc)

using synthetic digits (n=2); drop mnist_sample.npz beside script for real samples
=== MNIST tiny-ViT smoke ===
IMG=16 PATCH=4 T=8 D=8 classes=10
scales ACT/W/P=64.0/64.0/127.0
weights seeded; labels=[4, 5] (accuracy not expected yet)

--- image[0] label=4 ---
--- ref ---
ref pred=7 logits_q12[:4]=[-8, -74, 1, 101]

--- image[1] label=5 ---
--- ref ---
ref pred=1 logits_q12[:4]=[24, 89, -11, -48]

VIT REF-ONLY PASS (plumbing + synthetic/real sample path)
ref-only return 0


## Board: ref vs FPGA intermediates

In [3]:
rc = vit.run_vit_smoke(bit_path=BIT, seed=0, n=2)
assert rc == 0, "ViT smoke failed"
print("board vit return", rc)

using synthetic digits (n=2); drop mnist_sample.npz beside script for real samples
=== MNIST tiny-ViT smoke ===
IMG=16 PATCH=4 T=8 D=8 classes=10
scales ACT/W/P=64.0/64.0/127.0
weights seeded; labels=[4, 5] (accuracy not expected yet)


Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID=0x4E50554B version=0x00000300 features=0x00000003

--- image[0] label=4 ---
--- ref ---
ref pred=7 logits_q12[:4]=[-8, -74, 1, 101]
--- FPGA ---
hw  pred=7 logits_q12[:4]=[-8, -74, 1, 101]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=44  tol=1024
logits: PASS  max|err|=0  tol=1024

--- image[1] label=5 ---
--- ref ---
ref pred=1 logits_q12[:4]=[24, 89, -11, -48]
--- FPGA ---
hw  pred=1 logits_q12[:4]=[17, 92, -19, -43]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=39  tol=1024
logits: PASS  max|err|=12  tol=1024

ALL VIT PASS
board vit return 0
